<a href="https://colab.research.google.com/github/AntonyJohny/Innomatics_Research_Labs/blob/main/GenAi/Task_2/GenAI_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1: Install Dependencies**

In [ ]:
!pip install -qU langchain langchain-community

In [ ]:
# 1. Install necessary libraries for HuggingFace
!pip install -qU langchain-huggingface huggingface_hub

**2: API Key Configuration**

In [ ]:
import os
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpoint

# Safely retrieve your API key from Colab Secrets
try:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')
    print("✅ OpenAI API Key loaded successfully.")
except Exception as e:
    print("❌ API Key not found in Secrets. Please add it to the 'Keys' tab.")

✅ OpenAI API Key loaded successfully.

**3: Simple LCEL Chain**

In [ ]:
import os
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Setup Token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# 2. Initialize Model using the modern Endpoint class
# We use 'google/flan-t5-xxl' because it is highly reliable for free-tier RAG
repo_id = "google/flan-t5-xxl"

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    task="text2text-generation", # Explicitly set for FLAN-T5
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    pipeline_kwargs={
        "max_new_tokens": 250,
        "temperature": 0.5,
    },
)

# 3. Define the Prompt Template
prompt = ChatPromptTemplate.from_template("Explain the concept of {topic} in three bullet points.")

# 4. Create the Chain
chain = prompt | llm | StrOutputParser()

# 5. Run the Chain
print("--- Chain Output (Final Stable Implementation) ---")
try:
    print(chain.invoke({"topic": "Vector Embeddings"}))
except Exception as e:
    print(f"Implementation Note: {e}")

**4: State Management (Memory)**

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

# Alternative way to show state management
history = ChatMessageHistory()
history.add_user_message("Hi, my name is Antony.")
history.add_ai_message("Hello Antony!")

print("--- Chat History ---")
print(history.messages)

Output

--- Chat History ---
[HumanMessage(content='Hi, my name is Antony.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Antony!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

5: Advanced ReAct Agent

In [ ]:
# 1. Direct imports from the specific sub-modules
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents.react.agent import create_react_agent
from langchain.agents.agent import AgentExecutor
from langchain import hub

# 2. Ensure 'llm' from Cell 3 is active
tools = load_tools(["llm-math"], llm=llm)

# 3. Pull the reasoning prompt from the Hub
react_prompt = hub.pull("hwchase17/react")

# 4. Construct the Agent
agent = create_react_agent(llm, tools, react_prompt)

# 5. Initialize the Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("--- Agent Execution ---")
try:
    result = agent_executor.invoke({"input": "What is the square root of 529 multiplied by 5?"})
    print(f"\nFinal Result: {result['output']}")
except Exception as e:
    print(f"\nExecution Note: {e}")